In [ ]:
import os

# basic data engineering
import pandas as pd
import numpy as np
import scipy

# plotting
import matplotlib.pyplot as plt
import seaborn as sns

# db
import pymongo

# configs & other
import yaml
from tqdm.notebook import tqdm_notebook
from datetime import datetime
from time import time

from psynlig import pca_explained_variance_bar

# utils processing
from utils import sliding_window_pd
from utils import apply_filter
from utils import filter_instances
from utils import flatten_instances_df
from utils import df_rebase
from utils import rename_df_column_values

# utils features
from utils_features import extract_features_from_windows

# utils visualization
from utils_visual import plot_instance_time_domain
from utils_visual import plot_instance_3d
from utils_visual import plot_np_instance
from utils_visual import plot_heatmap
from utils_visual import plot_scatter_pca


Start time of execution

In [ ]:
time_start = time()

**Load Configuation**

In [ ]:
config_path = os.path.join(os.getcwd(), "config.yml")
with open(config_path, "r") as f:
    config = yaml.load(f, Loader=yaml.FullLoader)

In [ ]:
client = pymongo.MongoClient(config["client"])

In [ ]:
db = client[config["db"]]
collection = db[config["col"]]

In [ ]:
found_keys = collection.distinct("_id")
print("Total number of documents in the collection:", len(found_keys))

**Load data**

In [ ]:
documents = collection.find({"_id": {"$in": found_keys}})
rows = []
for doc in documents:
    n_samples = len(doc["data"]["acc_x"])
    for i in range(n_samples):
        rows.append({
            "acc_x": float(doc["data"]["acc_x"][i]),
            "acc_y": float(doc["data"]["acc_y"][i]),
            "acc_z": float(doc["data"]["acc_z"][i]),
            "gyr_x": float(doc["data"]["gyr_x"][i]),
            "gyr_y": float(doc["data"]["gyr_y"][i]),
            "gyr_z": float(doc["data"]["gyr_z"][i]),
            "gesture_id": doc["gesture_id"],
            "hand": int(doc["hand"]),
            "sr": int(doc["sr"]),
            "user": doc["user"],
        })

dataframe = pd.DataFrame(rows)
# Should be float/int
print(dataframe[["acc_x", "acc_y", "acc_z", "gyr_x", "gyr_y", "gyr_z", "gesture_id", "hand", "sr", "user"]].dtypes)
print(dataframe[:5])

## Exploratory Data Analysis (EDA)

In [ ]:
# DEBUG: Simplify the problem by removing Left/Right gestures
gestures_to_keep = ["swipe-up-thumb", "swipe-down-thumb", "texting-two-hands"]
dataframe = dataframe[dataframe["gesture_id"].isin(gestures_to_keep)]

print(f"Remaining gestures: {dataframe.gesture_id.unique()}")
print(f"New dataset size: {dataframe.shape[0]} rows")


### Step 1: Clean and validate sensor columns

In [ ]:
sensor_cols = ["acc_x", "acc_y", "acc_z", "gyr_x", "gyr_y", "gyr_z"]

# Force numeric and coerce invalid values to NaN
dataframe[sensor_cols] = dataframe[sensor_cols].apply(pd.to_numeric, errors="coerce")

missing_before = dataframe[sensor_cols].isna().sum().sum()

# Interpolate missing values and fill any remaining edge NaNs
interpolated = dataframe[sensor_cols].interpolate(limit_direction="both")
dataframe[sensor_cols] = interpolated.fillna(method="bfill").fillna(method="ffill")

missing_after = dataframe[sensor_cols].isna().sum().sum()

print(f"Missing sensor values before cleaning: {missing_before}")
print(f"Missing sensor values after cleaning: {missing_after}")
print("Cleaned dataset shape:", dataframe.shape)

In [ ]:
# Step 1.1: Cap Gyroscope Outliers (1st and 99th percentile)
# We calculate bounds from training users (01, 02) to prevent leakage
train_users = ["01", "02"]
gyr_cols = ["gyr_x", "gyr_y", "gyr_z"]

# Calculate bounds
train_gyr = dataframe[dataframe["user"].isin(train_users)][gyr_cols]
gyr_low = train_gyr.quantile(0.01)
gyr_high = train_gyr.quantile(0.99)

# Apply clipping to the entire dataset
dataframe[gyr_cols] = dataframe[gyr_cols].clip(lower=gyr_low, upper=gyr_high, axis=1)

print("Gyroscope outliers capped using 1st and 99th percentiles of training users.")
print(f"Lower bounds:\n{gyr_low}")
print(f"Upper bounds:\n{gyr_high}")


### Step 2: Dataset overview

In [ ]:
print("Dataset shape:", dataframe.shape)
print("\nFirst few rows:")
print(dataframe.head())
print("\nDataset info:")
print(dataframe.info())
print("\nDataset statistics:")
print(dataframe.describe())

### Step 3: Time-length barplot for each class

In [ ]:
# Calculate time-length for each gesture class
# Time in seconds = number of samples / sampling rate
time_length_per_gesture = dataframe.groupby("gesture_id").size() / dataframe["sr"].iloc[0]

# Create barplot
plt.figure(figsize=(12, 6))
time_length_per_gesture.plot(kind="bar", color="steelblue", edgecolor="black")
plt.title("Time-length of Instances per Gesture Class", fontsize=16, fontweight="bold")
plt.xlabel("Gesture Class", fontsize=12)
plt.ylabel("Total Time (seconds)", fontsize=12)
plt.xticks(rotation=45, ha="right")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

print("Time-length per gesture class:")
print(time_length_per_gesture)

### Step 4: Time-series segmentation with sliding window

In [ ]:
# Load sliding window parameters from config
ws = config["sliding_window"]["ws"]  # window size in samples
overlap = config["sliding_window"]["overlap"]  # overlap in samples
w_type = config["sliding_window"]["w_type"]  # window type (e.g., "hann")
w_center = config["sliding_window"]["w_center"]  # center window

print("Sliding window parameters:")
print(f"  Window size (samples): {ws}")
print(f"  Overlap (samples): {overlap}")
print(f"  Window type: {w_type}")
print(f"  Center window: {w_center}")

# Group by subject and gesture to avoid mixing users
windows_per_group = {}
for (user_id, gesture_id), group_df in dataframe.groupby(["user", "gesture_id"], sort=False):
    gesture_df = group_df.reset_index(drop=True)
    windows = sliding_window_pd(
        gesture_df,
        ws=ws,
        overlap=overlap,
        w_type=w_type,
        w_center=w_center,
        print_stats=True
    )
    group_key = (str(user_id), str(gesture_id))
    windows_per_group[group_key] = windows
    print(f"\nuser={user_id} | gesture={gesture_id}: Generated {len(windows)} windows")

# Backward-compatible alias (if later cells expect this name)
windows_per_gesture = windows_per_group

In [ ]:
# from utils_segmentation import extract_peak_centered_windows
# 
# # SMART WINDOWING: Center windows on movement peaks
# # This removes the "noise" and ensures every window contains a gesture
# ws_smart = 256  # 2.5 seconds at 100Hz
# threshold_smart = 1.15  # Trigger when acceleration magnitude > 1.15g
# 
# smart_windows_per_gesture = {}
# 
# for (user_id, gesture_id), group_df in dataframe.groupby(["user", "gesture_id"], sort=False):
#     gesture_df = group_df.reset_index(drop=True)
#     
#     # Extract only the "Action" windows
#     windows = extract_peak_centered_windows(
#         gesture_df, 
#         ws=ws_smart, 
#         threshold=threshold_smart
#     )
#     
#     group_key = (str(user_id), str(gesture_id))
#     smart_windows_per_gesture[group_key] = windows
#     print(f"user={user_id} | gesture={gesture_id}: Extracted {len(windows)} ACTION windows")
# 
# # Use these smart windows for the rest of the pipeline
# windows_per_gesture = smart_windows_per_gesture


### Step 5: Barplot of instance count after windowing

In [ ]:
# Count windows per user+gesture group
window_counts = {key: len(windows) for key, windows in windows_per_gesture.items()}

# Create barplot
plt.figure(figsize=(12, 6))
labels = [f"user={user} | gesture={gesture}" for user, gesture in window_counts.keys()]
counts = list(window_counts.values())
plt.bar(labels, counts, color="coral", edgecolor="black")
plt.title("Number of Windowed Instances per User/Gesture", fontsize=16, fontweight="bold")
plt.xlabel("User/Gesture", fontsize=12)
plt.ylabel("Window Count", fontsize=12)
plt.xticks(rotation=45, ha="right")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

print("Window count per user/gesture:")
for (user, gesture), count in window_counts.items():
    print(f"  user={user} | gesture={gesture}: {count} windows")

In [ ]:
# Check for missing values
print("=" * 60)
print("MISSING VALUES ANALYSIS")
print("=" * 60)
missing_values = dataframe.isnull().sum()
if missing_values.sum() == 0:
    print("No missing values found in the dataset")
else:
    print("Missing values per column:")
    print(missing_values[missing_values > 0])

# Outlier detection using IQR method (Interquartile Range)
print("\n" + "=" * 60)
print("OUTLIER DETECTION (IQR Method)")
print("=" * 60)

sensor_cols = ["acc_x", "acc_y", "acc_z", "gyr_x", "gyr_y", "gyr_z"]
outliers_count = {}

for col in sensor_cols:
    Q1 = dataframe[col].quantile(0.25)
    Q3 = dataframe[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = (dataframe[col] < lower_bound) | (dataframe[col] > upper_bound)
    outliers_count[col] = outliers.sum()

    print(f"\n{col}:")
    print(f"  Q1: {Q1:.4f}, Q3: {Q3:.4f}, IQR: {IQR:.4f}")
    print(f"  Lower bound: {lower_bound:.4f}, Upper bound: {upper_bound:.4f}")
    print(f"  Outliers detected: {outliers.sum()} ({100*outliers.sum()/len(dataframe):.2f}%)")

# Visualize outlier distribution
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for idx, col in enumerate(sensor_cols):
    axes[idx].boxplot(dataframe[col].dropna(), vert=True)
    axes[idx].set_title(f"{col} - Outliers: {outliers_count[col]}", fontweight="bold")
    axes[idx].set_ylabel("Value", fontsize=10)
    axes[idx].grid(alpha=0.3)

plt.suptitle("Outlier Detection via Boxplots (IQR Method)", fontsize=14, fontweight="bold", y=1.00)
plt.tight_layout()
plt.show()

print("\n" + "=" * 60)
print("SUMMARY STATISTICS")
print("=" * 60)
print(f"Total samples: {len(dataframe)}")
print(f"Total missing values: {missing_values.sum()}")
print(f"Total outliers detected: {sum(outliers_count.values())}")

### Step 6: Outlier detection (train users only)

### Step 6: Filter data with low-pass filter

In [ ]:
# Load filter parameters from config
filter_order = config["filter"]["order"]
filter_wn = config["filter"]["wn"]  # normalized critical frequency
filter_type = config["filter"]["type"]

print("Filter parameters:")
print(f"  Order: {filter_order}")
print(f"  Critical frequency (normalized): {filter_wn}")
print(f"  Filter type: {filter_type}")

# Apply filter to all windows per gesture
filtered_windows_per_gesture = {}
for gesture_id, windows in windows_per_gesture.items():
    filtered_windows = filter_instances(
        windows,
        order=filter_order,
        wn=filter_wn,
        filter_type=filter_type
    )
    filtered_windows_per_gesture[gesture_id] = filtered_windows
    print(f"Filtered {len(filtered_windows)} windows for gesture '{gesture_id}'")

### Step 7: Visualize filter effect on signal

In [ ]:
# Select first gesture and first window for visualization
first_gesture = list(windows_per_gesture.keys())[0]
original_window = windows_per_gesture[first_gesture][0]
filtered_window = filtered_windows_per_gesture[first_gesture][0]

# Extract only the sensor columns (acc_x, acc_y, acc_z, gyr_x, gyr_y, gyr_z)
sensor_cols = ["acc_x", "acc_y", "acc_z", "gyr_x", "gyr_y", "gyr_z"]
original_sensors = original_window[sensor_cols]
filtered_sensors = filtered_window[sensor_cols]

# Plot before and after filtering
fig, axes = plt.subplots(2, 1, figsize=(16, 10))

# Original signal
axes[0].plot(original_sensors.values, linewidth=2)
axes[0].set_title(f"Original Signal - {first_gesture}", fontsize=14, fontweight="bold")
axes[0].set_ylabel("Amplitude", fontsize=12)
axes[0].legend(sensor_cols, loc="upper right")
axes[0].grid(alpha=0.3)

# Filtered signal
axes[1].plot(filtered_sensors.values, linewidth=2)
axes[1].set_title(f"Filtered Signal (Low-pass, cutoff={filter_wn}) - {first_gesture}", fontsize=14, fontweight="bold")
axes[1].set_xlabel("Sample", fontsize=12)
axes[1].set_ylabel("Amplitude", fontsize=12)
axes[1].legend(sensor_cols, loc="upper right")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nVisualization of first window from '{first_gesture}' gesture:")
print(f"  Original shape: {original_sensors.shape}")
print(f"  Filtered shape: {filtered_sensors.shape}")

# Data Transformation and Feature Engineering

# Feature extraction (mean, std, min, max, etc.) for each window

In [ ]:
# # Data Augmentation: Synthetic Rotations
# # Since User 03 held the phone in Landscape, we will teach the model
# # that gestures can happen in different orientations by rotating Train data.

# def augment_rotations(windows_list):
#     augmented = []
#     for df in windows_list:
#         augmented.append(df) # Original (Portrait)
        
#         # Simulate 90-degree rotation (Landscape Right)
#         df_90 = df.copy()
#         df_90["acc_x"], df_90["acc_y"] = df["acc_y"], -df["acc_x"]
#         df_90["gyr_x"], df_90["gyr_y"] = df["gyr_y"], -df["gyr_x"]
#         augmented.append(df_90)
        
#         # Simulate -90-degree rotation (Landscape Left)
#         df_n90 = df.copy()
#         df_n90["acc_x"], df_n90["acc_y"] = -df["acc_y"], df["acc_x"]
#         df_n90["gyr_x"], df_n90["gyr_y"] = -df["gyr_y"], df["gyr_x"]
#         augmented.append(df_n90)
        
#     return augmented

# # Apply only to training users to prevent leakage
# train_users = ["01", "02"]
# for key in list(filtered_windows_per_gesture.keys()):
#     if key[0] in train_users:
#         original_count = len(filtered_windows_per_gesture[key])
#         filtered_windows_per_gesture[key] = augment_rotations(filtered_windows_per_gesture[key])
#         print(f"Augmented {key}: {original_count} -> {len(filtered_windows_per_gesture[key])} windows")

# print("\nData Augmentation Complete. Training set is now 3x larger and orientation-aware.")


In [ ]:
# Feature extraction from filtered windows
sensor_cols = ["acc_x", "acc_y", "acc_z", "gyr_x", "gyr_y", "gyr_z"]
X_features, y_gesture, y_user = extract_features_from_windows(filtered_windows_per_gesture, sensor_cols)

print(f"Windowed feature matrix shape: {X_features.shape}")
print("First 5 feature columns:", list(X_features.columns[:5]))
print(f"Gesture labels shape: {y_gesture.shape}")


In [ ]:
# Train/Test Split by User
train_users = ["01", "03"]
test_users = ["02"]

train_mask = y_user.isin(train_users)
test_mask = y_user.isin(test_users)

X_train, y_train = X_features[train_mask], y_gesture[train_mask]
X_test, y_test = X_features[test_mask], y_gesture[test_mask]

print(f"Training set: {X_train.shape[0]} windows (Users: {train_users})")
print(f"Test set: {X_test.shape[0]} windows (User: {test_users})")

# Drop noisy/redundant features (20 features)
cols_to_drop = [c for c in X_train.columns if any(s in c for s in ["_skew", "_kurt", "_sma", "_spec_energy"])]
X_train = X_train.drop(columns=cols_to_drop)
X_test = X_test.drop(columns=cols_to_drop)

print(f"Final feature count after manual pruning: {X_train.shape[1]}")


In [ ]:
from sklearn.preprocessing import StandardScaler

# Define Per-User Scaling function
# This normalizes each user to their own mean/std to remove personal style offsets
def apply_per_user_scaling(X, user_labels):
    X_scaled_list = []
    for user in user_labels.unique():
        user_mask = (user_labels == user)
        X_user = X[user_mask]
        scaler = StandardScaler()
        # Fit and transform only on this specific user
        X_user_scaled = pd.DataFrame(scaler.fit_transform(X_user), columns=X_user.columns, index=X_user.index)
        X_scaled_list.append(X_user_scaled)
    return pd.concat(X_scaled_list).sort_index()

# 1. Apply scaling individually for each user
X_train_scaled = apply_per_user_scaling(X_train, y_user[train_mask])
X_test_scaled = apply_per_user_scaling(X_test, y_user[test_mask])

print(f"Per-User Scaling complete.")
print(f"Final Training Shape: {X_train_scaled.shape}")
print(f"Final Test Shape: {X_test_scaled.shape}")

# 2. Sanity check: Mean of features for User 01 vs User 03 should now both be ~0
user01_mean = X_train_scaled[y_user[train_mask] == "01"].iloc[:, 0].mean()
user03_mean = X_test_scaled[y_user[test_mask] == "03"].iloc[:, 0].mean()
print(f"User 01 Feature Mean: {user01_mean:.4f}")
print(f"User 03 Feature Mean: {user03_mean:.4f}")


In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# 1. Initialize SVM Classifier
# RBF kernel is excellent for non-linear high-dimensional feature spaces like yours
clf = SVC(kernel="rbf", C=1.0, gamma="scale", random_state=42)

# 2. Train the model
clf.fit(X_train_scaled, y_train)

# 3. Predict on the test set (User 03)
y_pred = clf.predict(X_test_scaled)

# 4. Evaluation
print("="*60)
print("SVM CLASSIFICATION REPORT (User 03 - Unseen)")
print("="*60)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.2%}")
print("\n", classification_report(y_test, y_pred))

# 5. Confusion Matrix Visualization
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", 
            xticklabels=clf.classes_, yticklabels=clf.classes_)
plt.title("Confusion Matrix: SVM Gesture Classification")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()


In [ ]:
from sklearn.ensemble import RandomForestClassifier

# 1. Initialize Random Forest
# Trees are great at capturing the "threshold" logic of gestures
rf_clf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)

# 2. Train the model
rf_clf.fit(X_train_scaled, y_train)

# 3. Predict on the test set (User 03)
y_pred_rf = rf_clf.predict(X_test_scaled)

# 4. Evaluation
print("="*60)
print("RANDOM FOREST CLASSIFICATION REPORT (User 03)")
print("="*60)
print(f"Accuracy: {accuracy_score(y_test, y_pred_rf):.2%}")
print("\n", classification_report(y_test, y_pred_rf))

# 5. Feature Importance Visualization
importances = rf_clf.feature_importances_
# Get indices of the top 15 features
indices = np.argsort(importances)[-15:]

plt.figure(figsize=(12, 8))
plt.title("Top 15 Most Discriminative Features (Random Forest)", fontsize=14, fontweight="bold")
plt.barh(range(len(indices)), importances[indices], align="center", color="forestgreen")
plt.yticks(range(len(indices)), [X_train_scaled.columns[i] for i in indices])
plt.xlabel("Relative Importance")
plt.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# Orientation-Invariant Model (Magnitude Only)
# We only use features derived from acc_mag and gyr_mag
mag_cols = [c for c in X_train_scaled.columns if "mag" in c]

X_train_mag = X_train_scaled[mag_cols]
X_test_mag = X_test_scaled[mag_cols]

print(f"Training on {len(mag_cols)} orientation-invariant (magnitude) features: {mag_cols}")

# Train Random Forest on Magnitudes only
rf_mag = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf_mag.fit(X_train_mag, y_train)

# Predict on User 03
y_pred_mag = rf_mag.predict(X_test_mag)

# Evaluation
print("="*60)
print("MAGNITUDE-ONLY CLASSIFICATION REPORT (User 03 - Unseen)")
print("="*60)
print(f"Accuracy: {accuracy_score(y_test, y_pred_mag):.2%}")
print("\n", classification_report(y_test, y_pred_mag))


In [ ]:
# Diagnostic: Check performance on the TRAINING set (Users 01 & 02)
y_train_pred = rf_clf.predict(X_train_scaled)

print("="*60)
print("DIAGNOSTIC: PERFORMANCE ON TRAINING DATA (Users 01 & 02)")
print("="*60)
print(f"Training Accuracy: {accuracy_score(y_train, y_train_pred):.2%}")
print("\n", classification_report(y_train, y_train_pred))


In [ ]:
# Visual Audit: Compare User 01 vs User 03 (Swipe Up)
# This will help us see if the axes are swapped or if the styles are too different
gesture_name = "swipe-up-thumb"

# Find the first window for each user
win01 = None
win03 = None

for (uid, gname), windows in filtered_windows_per_gesture.items():
    if gname == gesture_name:
        if uid == "01": win01 = windows[0]
        if uid == "03": win03 = windows[0]

if win01 is not None and win03 is not None:
    fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharey=True)
    
    # User 01 Plot
    axes[0].plot(win01[["acc_x", "acc_y", "acc_z"]].values)
    axes[0].set_title(f"USER 01 (Train) - {gesture_name}", fontsize=14, fontweight="bold")
    axes[0].legend(["acc_x", "acc_y", "acc_z"], loc="upper right")
    axes[0].set_ylabel("Acceleration (g)")
    axes[0].grid(alpha=0.3)
    
    # User 03 Plot
    axes[1].plot(win03[["acc_x", "acc_y", "acc_z"]].values)
    axes[1].set_title(f"USER 03 (Test) - {gesture_name}", fontsize=14, fontweight="bold")
    axes[1].legend(["acc_x", "acc_y", "acc_z"], loc="upper right")
    axes[1].set_ylabel("Acceleration (g)")
    axes[1].set_xlabel("Samples")
    axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print(f"Could not find windows for {gesture_name}. Check if the users and classes exist.")


In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

# 1. Combine everything (Users 01, 02, and 03 are now in the same pool)
X_full = pd.concat([X_train, X_test])
y_full = pd.concat([y_train, y_test])

# 2. Apply Global Scaling
scaler_global = StandardScaler()
X_full_scaled = scaler_global.fit_transform(X_full)

# 3. Initialize 5-Fold Cross Validation
# This will split the mixed data into 5 parts and test each part once
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 4. Evaluate the Random Forest
cv_scores = cross_val_score(rf_clf, X_full_scaled, y_full, cv=skf, scoring="accuracy")

print("="*60)
print("5-FOLD CROSS VALIDATION RESULTS (Mixed Users)")
print("="*60)
print(f"Fold Accuracies: {cv_scores}")
print(f"Average Accuracy: {cv_scores.mean():.2%}")
print(f"Stability (Std Dev): {cv_scores.std():.2%}")

if cv_scores.mean() > 0.80:
    print("\nCONCLUSION: The features are GREAT, but the model needs examples from every user to work well.")
else:
    print("\nCONCLUSION: Even with mixed data, the accuracy is low. We need better features or cleaner data.")


In [ ]:
from sklearn.manifold import TSNE

# T-SNE Lie Detector: Can we actually see the clusters?
# This squashes your 48 features into 2D so you can see if they are separable
print("Running t-SNE... (this might take a minute)")

tsne = TSNE(n_components=2, perplexity=30, n_iter=1000, random_state=42)
X_tsne = tsne.fit_transform(X_train_scaled)

# Plot the clusters
plt.figure(figsize=(12, 8))
sns.scatterplot(
    x=X_tsne[:, 0], y=X_tsne[:, 1], 
    hue=y_train, 
    palette="Set1", 
    s=60, 
    alpha=0.6,
    edgecolor="w"
)

plt.title("t-SNE Visualization: Gesture Cluster Separability", fontsize=15, fontweight="bold")
plt.xlabel("t-SNE Component 1")
plt.ylabel("t-SNE Component 2")
plt.legend(title="Gestures", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.grid(alpha=0.2)
plt.show()

print("\nHOW TO READ THIS PLOT:")
print("1. Clear Islands = Excellent features. Just need a better model.")
print("2. Overlapping Clouds = Too much noise. Smart Windowing is REQUIRED.")
print("3. One big mess = The features are not capturing the gestures at all.")


In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC

# 1. SVM GRID SEARCH
param_grid_svm = {
    "C": [0.1, 1, 10, 100],
    "gamma": ["scale", "auto", 0.01, 0.1],
    "kernel": ["rbf", "poly"]
}

print("Running SVM Grid Search on Magnitude Features...")
grid_svm = GridSearchCV(SVC(probability=True), param_grid_svm, cv=3, scoring="accuracy", verbose=1)
grid_svm.fit(X_train_mag, y_train)

print(f"Best SVM Params: {grid_svm.best_params_}")
best_svm = grid_svm.best_estimator_

# Evaluate Best SVM
y_pred_svm = best_svm.predict(X_test_mag)
print("\n" + "="*40)
print("TUNED SVM MAGNITUDE-ONLY ACCURACY: ", accuracy_score(y_test, y_pred_svm))
print("="*40)

# 2. RANDOM FOREST GRID SEARCH
param_grid_rf = {
    "n_estimators": [100, 200],
    "max_depth": [None, 10, 20],
    "criterion": ["gini", "entropy"]
}

print("\nRunning Random Forest Grid Search...")
grid_rf = GridSearchCV(RandomForestClassifier(random_state=42, n_jobs=-1), param_grid_rf, cv=3, scoring="accuracy", verbose=1)
grid_rf.fit(X_train_mag, y_train)

print(f"Best RF Params: {grid_rf.best_params_}")
best_rf = grid_rf.best_estimator_

# Evaluate Best RF
y_pred_rf = best_rf.predict(X_test_mag)
print("\n" + "="*40)
print("TUNED RF MAGNITUDE-ONLY ACCURACY: ", accuracy_score(y_test, y_pred_rf))
print("="*40)


In [ ]:
# MINIMALIST EXPLAINABLE MODEL
# Features chosen specifically by you for their domain logic

minimalist_features = [
    "acc_y_mean",
    "acc_y_min",
    "acc_mag_range",
    "acc_mag_std",
    "acc_z_zcr",
    "acc_z_mean",
    "gyr_mag_max"
]

# Check if features are ready
missing = [f for f in minimalist_features if f not in X_train.columns]
if missing:
    print(f"WARNING: Features {missing} are missing. PLEASE RE-RUN FEATURE EXTRACTION CELL.")
else:
    X_train_min = X_train_scaled[minimalist_features]
    X_test_min = X_test_scaled[minimalist_features]

    print(f"Training on your curated list of {len(minimalist_features)} features.")

    # Train
    rf_min = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    rf_min.fit(X_train_min, y_train)

    # Evaluate
    y_pred_min = rf_min.predict(X_test_min)
    print("="*60)
    print("CURATED MINIMALIST MODEL REPORT")
    print("="*60)
    print(f"Accuracy: {accuracy_score(y_test, y_pred_min):.2%}")
    print("\n", classification_report(y_test, y_pred_min))

    # Importance
    importances = rf_min.feature_importances_
    plt.figure(figsize=(10, 6))
    plt.title("Curated Feature Importance")
    plt.barh(minimalist_features, importances)
    plt.xlabel("Importance Score")
    plt.show()
